In [ ]:
# Who's the Best Pitcher -- GENERATIVE answering, not index-picking
#
# WHY THIS EXISTS. Everything built so far asks the model "here are 8 cryptic
# field paths, reply with a number". That is index-picking, and it has a hard
# ceiling: the truth is in our 8-candidate shortlist only ~86% of the time
# (79/92 = 0.857), and the 4B provably chose WORSE than the solver -- loosening
# the gate lost monotonically, 63 -> 62 -> 60.
#
# The leaderboard says that whole framing is wrong. Multiply by 92:
#     us                     0.68478 = 63/92
#     seven-team cluster     0.79347 = 73/92     <- almost certainly the starter
#     record                 ~0.95   = ~87/92
# A record at ~87/92 disproves the 79/92 "ceiling" -- that number described OUR
# candidate generation, not the task. Teams doing GENERATIVE QA over readable
# context are 10-24 questions ahead of an elaborate index-picker.
#
# So: give the model the question plus the candidates rendered READABLY, let it
# ANSWER in its own words, then snap that answer back to a stored value.
#
# Exact-match safety is preserved by construction: we never emit model text. The
# generated answer is only used to SELECT which stored value to copy verbatim.
# "123.4500" stays "123.4500" -- the thing that makes generation dangerous here
# is avoided entirely.
import glob, json, os, re, time
import numpy as np, pandas as pd, torch

CAND = (sorted(glob.glob("/kaggle/input/**/candidates200.json", recursive=True))
        or sorted(glob.glob("/kaggle/input/**/candidates.json", recursive=True)))
assert CAND, "attach candidates.json as a Dataset"
rows = json.load(open(CAND[0]))
print(f"{len(rows)} questions from {CAND[0]}")

# SMOKE=5 runs five questions and prints each raw generation, so a broken
# generation path costs two minutes instead of an hour. The last full run
# returned FALLBACK-top1 on all 220 -- every answer discarded -- and that was
# only visible AFTER 14 minutes. Set SMOKE=0 for the real run.
SMOKE = 5
if SMOKE:
    rows = rows[:SMOKE]
    print(f"*** SMOKE TEST: {len(rows)} questions ***")

# SANITISER REMOVED: measured on the leaderboard it changed 6 top-1 answers and
# cost a net question (63/92 -> 62/92). The 'guaranteed wrong' answers it
# replaced were often replaced by other wrong answers (RBIs '1' -> '43827',
# which is attendance), so it traded known-wrong for unknown-wrong.
TEST = sorted(glob.glob("/kaggle/input/**/test.csv", recursive=True))
print("test.csv:", TEST[0] if TEST else "(not needed)")

In [ ]:
# The starter's `pip install git+...@v4.49.0-Gemma-3` dates from before Gemma-3
# shipped in a transformers release. Kaggle's image now has 4.53.3 with native
# support, so that line DOWNGRADES to a dev build and leaves the install
# inconsistent -- which is what raises ImportError inside AutoProcessor.
# Only install if Gemma3 is genuinely missing.
import importlib, transformers
print('transformers', transformers.__version__)
try:
    from transformers import Gemma3ForConditionalGeneration
    print('Gemma3 available -- no install needed')
except Exception as e:
    print('Gemma3 missing:', e)
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3'])
    print('INSTALLED -- now RESTART THE KERNEL and run from cell 1 again')
    raise SystemExit('restart kernel')


In [ ]:
from transformers import AutoProcessor, AutoTokenizer, Gemma3ForConditionalGeneration, BitsAndBytesConfig

dirs = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
        if os.path.exists(os.path.join(d, "config.json"))]
print("model dirs:"); [print("   ", d) for d in dirs]
pref = [d for d in dirs if "27b" in d.lower()] or [d for d in dirs if "gemma" in d.lower()] or dirs
MODEL = pref[0].rstrip("/"); print("using:", MODEL)

CUDA = torch.cuda.is_available()
BIG  = "27b" in MODEL.lower()
if BIG and CUDA:
    # 27B is 54 GB in fp16 against a T4's 16 GB; NF4 brings it to ~14 GB.
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_use_double_quant=True)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL, device_map="auto", quantization_config=q).eval()
else:
    # is_bf16_supported() counts EMULATED support and returns True on a T4
    # (Turing, no native bf16). Gate on compute capability instead.
    dt = torch.float16
    if CUDA and torch.cuda.get_device_capability()[0] >= 8: dt = torch.bfloat16
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL, device_map="auto" if CUDA else None, torch_dtype=dt).eval()
# AutoProcessor pulls in Gemma3ImageProcessor, and this task is TEXT ONLY --
# that import path is exactly where the ImportError came from. The tokenizer
# carries the chat template, which is all we need.
# AutoProcessor, like the starter. The ImportError that made me switch to
# AutoTokenizer came from the transformers DOWNGRADE, which is long removed --
# and the tokenizer path generates 96 tokens that decode to nothing.
try:
    processor = AutoProcessor.from_pretrained(MODEL)
    USE_PROC = True
    print('using AutoProcessor')
except Exception as e:
    print('AutoProcessor failed:', repr(e)[:200])
    processor = AutoTokenizer.from_pretrained(MODEL); USE_PROC = False
if getattr(processor, 'chat_template', None) is None:
    _ct = os.path.join(MODEL, 'chat_template.json')
    if os.path.exists(_ct):
        processor.chat_template = json.load(open(_ct))['chat_template']
        print('chat template loaded from chat_template.json')
assert getattr(processor, 'chat_template', None), 'no chat template found'
print("loaded on", "cuda" if CUDA else "cpu")

In [ ]:
SYSTEM = ("You answer questions about a baseball statistics database. You are shown "
          "the database fields that might contain the answer, each with its location "
          "and its exact stored value. Reply with ONLY the value that answers the "
          "question -- copy it exactly as shown, no units, no words, no explanation.")

# Rung 2 of the escalation: the shortlist holds the truth only ~86% of the
# time, which caps this whole design at 79/92. Widening K raises that ceiling
# at the cost of a longer prompt -- generation tolerates that far better than
# index-picking did. Bump this first if the run shows signal.
# 150, not 8. The truth is in the top-8 only ~86% of the time, which caps ANY
# selection method at 79/92 -- and two independent ones have now landed below
# the solver. 150 leaves x ~136 chars is ~5k tokens, well inside context.
TOPK = 150

def render(row, k=None):
    k = TOPK if k is None else k
    """Readable context. The path is spelled out in words, which is the whole
    point: 'onbase > s' means nothing, 'on base > singles' does."""
    out = []
    for i, c in enumerate(row["candidates"][:k], 1):
        loc = c["text"].replace(" > ", " / ")
        out.append(f"{i}. {loc}\n   value: {c['value']}")
    return "\n".join(out)

import gc

import gc

# The chat template is the remaining unknown: we hand a TOKENIZER the PROCESSOR's
# template, and five runs have returned empty strings. Build Gemma's turn format
# by hand instead -- it is three literal markers, and it removes the template
# from the equation entirely.
def build_prompt(system, user):
    return (f"<start_of_turn>user\n{system}\n\n{user}<end_of_turn>\n"
            f"<start_of_turn>model\n")

@torch.no_grad()
def answer(row, max_new=96):
    for attempt, (k, mx) in enumerate([(TOPK, max_new), (8, 64), (4, 48)]):
        inp = gen = None
        try:
            u = (f"Question: {row['question']}\n\nDatabase fields:\n"
                 f"{render(row, k)}\n\nWhich value answers the question? "
                 f"Reply with the value only.")
            if USE_PROC:
                msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                        {"role": "user",   "content": [{"type": "text", "text": u}]}]
                inp = processor.apply_chat_template(
                    msgs, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt").to(model.device)
            else:
                inp = processor(build_prompt(SYSTEM, u), return_tensors="pt").to(model.device)
            n = inp["input_ids"].shape[-1]
            gen = model.generate(**inp, max_new_tokens=mx, do_sample=False,
                                 use_cache=True)
            full = processor.decode(gen[0], skip_special_tokens=True)
            out  = processor.decode(gen[0][n:], skip_special_tokens=True).strip()
            if SMOKE or not hasattr(answer, "_shown"):
                answer._shown = True
                print("=== PROMPT ===\n" + u[:400])
                print(f"=== in_tokens={n} out_tokens={gen.shape[-1]} "
                      f"new={gen.shape[-1]-n} ===")
                print("=== SLICED  ===", repr(out))
                print("=== FULL TAIL ===", repr(full[-200:]))
                print("=== CANDIDATES ===", [c["value"] for c in row["candidates"]][:8])
            # if the slice is empty but the model clearly spoke, fall back to the
            # tail of the full decode -- generate() does not always return the
            # prompt, and slicing past the end silently yields ""
            if not out and gen.shape[-1] <= n:
                out = full.strip()
            return out
        except torch.cuda.OutOfMemoryError:
            if attempt == 2: return ""
        finally:
            del inp, gen
            torch.cuda.empty_cache(); gc.collect()
    return ""


In [ ]:
def snap(text, row):
    """Map the model's words back to a STORED value. Never emit model text.

    Exact string match is the metric, so a generated '123.45' would lose to the
    stored '123.4500'. Matching back guarantees we always ship the source bytes.
    Longest match wins, so '0.267' is preferred over the '0' inside it.
    """
    vals = [c["value"] for c in row["candidates"]]
    t = text.strip()
    for v in sorted(set(vals), key=len, reverse=True):      # exact, then contained
        if t == str(v): return v, "exact"
    for v in sorted(set(vals), key=len, reverse=True):
        if str(v) and str(v) in t: return v, "contained"
    norm = re.sub(r"[^0-9a-z.]", "", t.lower())
    for v in sorted(set(vals), key=len, reverse=True):
        if norm and re.sub(r"[^0-9a-z.]", "", str(v).lower()) == norm: return v, "normalised"
    return (vals[0] if vals else "0"), "FALLBACK-top1"

picks, how = [], []
t0 = time.time()
for i, r in enumerate(rows):
    if not r["candidates"]:
        picks.append("no answer"); how.append("empty"); continue
    v, mode = snap(answer(r), r)
    picks.append(v); how.append(mode)
    if (i + 1) % 20 == 0:
        rate = (i + 1) / (time.time() - t0)
        print(f"  {i+1}/{len(rows)}  {rate:.2f} q/s  ETA {(len(rows)-i-1)/rate/60:.0f} min", flush=True)
print(f"done in {(time.time()-t0)/60:.1f} min")

import collections
print("\nhow the answer was matched back:", dict(collections.Counter(how)))
base = [r["candidates"][0]["value"] if r["candidates"] else "no answer" for r in rows]
moved = sum(1 for a, b in zip(picks, base) if a != b)
print(f"differs from the solver's top-1 on {moved}/{len(rows)} "
      f"(~{moved*92/220:.1f} scored questions)")

In [ ]:
sub = pd.DataFrame({"ID": [r["ID"] for r in rows], "ANSWER": picks})
assert len(sub) == 220 and not sub.ANSWER.isna().any()
sub.set_index("ID").to_csv("/kaggle/working/submission_generative.csv")
print("wrote submission_generative.csv")

# Also ship a CONSERVATIVE variant: keep the model's answer only where it matched
# a stored value exactly or by containment, and fall back to the solver elsewhere.
# Every confidence gate widened on this competition has lost, so the tight
# version is the one to submit first.
tight = [p if h in ("exact", "contained") else b
         for p, h, b in zip(picks, how, base)]
pd.DataFrame({"ID": [r["ID"] for r in rows], "ANSWER": tight}).set_index("ID") \
  .to_csv("/kaggle/working/submission_generative_tight.csv")
print("wrote submission_generative_tight.csv  "
      f"({sum(1 for a,b in zip(tight,base) if a!=b)} differ from solver top-1)")